# Seldon Core V2 — Local Setup Helper

This notebook makes it easy to (re)connect a local shell / Jupyter kernel to a Seldon Core v2 deployment running in a Kubernetes cluster (typically a `kind` cluster).

It performs the following steps:

1. Detect cluster reachability (`kubectl`).
2. Discover the Seldon scheduler & mesh services and their ports.
3. Kill any stale `kubectl port-forward` processes on the local ports we want to use.
4. Start fresh background `port-forward`s for the scheduler (gRPC) and the inference mesh (HTTP).
5. Export the env vars the `seldon` CLI expects (`SELDON_SCHEDULE_HOST`, `SELDON_INFER_HOST`).
6. Verify everything end-to-end with `seldon model list` and a raw `curl`.

> ⚠️ The `seldon` CLI auto-prepends `http://` to `SELDON_INFER_HOST`, so the env var must be set as `host:port` (no scheme).

## Configuration

Adjust these if your cluster layout differs.

In [ ]:
import os

# Kubernetes namespace where Seldon Core v2 is installed
NAMESPACE = "seldon-mesh"

# Service names (defaults from the seldon-core helm chart)
SCHEDULER_SVC = "seldon-scheduler"
MESH_SVC      = "seldon-mesh"

# Local ports to forward to. Use uncommon ports to avoid collisions.
SCHED_LOCAL_PORT = 9104   # scheduler gRPC (CLI control plane)
INFER_LOCAL_PORT = 9100   # inference HTTP (data plane via envoy)

# Service ports inside the cluster (do not change unless you customised the chart)
SCHED_SVC_PORT   = 9004   # 'scheduler' (plaintext gRPC)
MESH_SVC_PORT    = 80     # seldon-mesh HTTP

print(f"namespace          : {NAMESPACE}")
print(f"scheduler svc      : svc/{SCHEDULER_SVC}  {SCHED_LOCAL_PORT} -> {SCHED_SVC_PORT}")
print(f"inference mesh svc : svc/{MESH_SVC}  {INFER_LOCAL_PORT} -> {MESH_SVC_PORT}")

## 1. Cluster reachability

In [ ]:
import subprocess, shutil, sys

def run(cmd, check=False, capture=True):
    """Run a shell command, return (rc, stdout, stderr)."""
    p = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}\n{p.stderr}")
    return p.returncode, (p.stdout or "").strip(), (p.stderr or "").strip()

for tool in ("kubectl", "seldon", "curl"):
    path = shutil.which(tool)
    print(f"{tool:<8} -> {path or 'NOT FOUND'}")

rc, out, err = run("kubectl cluster-info --request-timeout=5s")
if rc != 0:
    print("\nERROR: cannot reach the cluster:\n", err)
else:
    print("\n" + out.splitlines()[0])

## 2. Service discovery

In [ ]:
import json

def get_svc_ports(ns, svc):
    rc, out, err = run(f"kubectl -n {ns} get svc {svc} -o json")
    if rc != 0:
        print(f"  svc/{svc} NOT FOUND in ns/{ns}: {err}")
        return None
    obj = json.loads(out)
    return [(p.get("name", "?"), p["port"], p.get("targetPort"), p.get("protocol", "TCP"))
            for p in obj["spec"]["ports"]]

for svc in (SCHEDULER_SVC, MESH_SVC):
    print(f"svc/{svc}:")
    ports = get_svc_ports(NAMESPACE, svc)
    if ports:
        for name, port, target, proto in ports:
            print(f"  {name:<16} {proto:<4} {port}  -> targetPort {target}")
    print()

Validate that the service ports we plan to forward actually exist.

In [ ]:
def has_port(ns, svc, port):
    ports = get_svc_ports(ns, svc) or []
    return any(p[1] == port for p in ports)

ok = True
if not has_port(NAMESPACE, SCHEDULER_SVC, SCHED_SVC_PORT):
    print(f"WARN: svc/{SCHEDULER_SVC} does not expose port {SCHED_SVC_PORT}")
    ok = False
if not has_port(NAMESPACE, MESH_SVC, MESH_SVC_PORT):
    print(f"WARN: svc/{MESH_SVC} does not expose port {MESH_SVC_PORT}")
    ok = False
print("all required service ports present" if ok else "please update the config cell")

## 3. Clean up stale port-forwards & local listeners

In [ ]:
def port_in_use(port):
    rc, out, _ = run(f"ss -tlnH 'sport = :{port}'")
    return bool(out.strip())

def kill_port_forward(svc, local_port):
    """Kill kubectl port-forward processes targeting this svc or local port."""
    pattern = f"kubectl.*port-forward.*({svc}|:{local_port}[^0-9])"
    rc, out, _ = run(f"pgrep -af '{pattern}'")
    if not out:
        return 0
    n = 0
    for line in out.splitlines():
        pid = line.split()[0]
        run(f"kill {pid} 2>/dev/null")
        n += 1
    return n

for svc, lport in [(SCHEDULER_SVC, SCHED_LOCAL_PORT), (MESH_SVC, INFER_LOCAL_PORT)]:
    killed = kill_port_forward(svc, lport)
    print(f"killed {killed} stale port-forward(s) for svc/{svc} on local :{lport}")

# Give the OS a moment to release the sockets
import time; time.sleep(2)
for p in (SCHED_LOCAL_PORT, INFER_LOCAL_PORT):
    print(f"port {p} in use: {port_in_use(p)}")

## 4. Start fresh port-forwards (background)

In [ ]:
import tempfile, time

LOG_DIR = tempfile.gettempdir()

def start_port_forward(ns, svc, local_port, svc_port):
    log = f"{LOG_DIR}/pf-{svc}.log"
    # nohup so it survives kernel restarts; & to background
    cmd = (
        f"nohup kubectl -n {ns} port-forward svc/{svc} "
        f"{local_port}:{svc_port} > {log} 2>&1 &"
    )
    subprocess.run(cmd, shell=True, executable="/bin/bash")
    return log

log_sched = start_port_forward(NAMESPACE, SCHEDULER_SVC, SCHED_LOCAL_PORT, SCHED_SVC_PORT)
log_mesh  = start_port_forward(NAMESPACE, MESH_SVC,      INFER_LOCAL_PORT, MESH_SVC_PORT)

# Wait until both ports are listening (max 10s)
deadline = time.time() + 10
while time.time() < deadline:
    if port_in_use(SCHED_LOCAL_PORT) and port_in_use(INFER_LOCAL_PORT):
        break
    time.sleep(0.5)

print(f"scheduler  log : {log_sched}  listening={port_in_use(SCHED_LOCAL_PORT)}")
print(f"mesh       log : {log_mesh}   listening={port_in_use(INFER_LOCAL_PORT)}")

if not (port_in_use(SCHED_LOCAL_PORT) and port_in_use(INFER_LOCAL_PORT)):
    print("\nPort-forwards did not come up. Last lines of logs:")
    for log in (log_sched, log_mesh):
        print(f"--- {log} ---")
        run(f"tail -n 20 {log}", capture=False)

## 5. Export env vars for the `seldon` CLI

These are set for the current Python process so `!seldon ...` magic in other notebooks (run by the same kernel) picks them up.

**Important:** `SELDON_INFER_HOST` must be `host:port` (no `http://` prefix) — the CLI prepends the scheme itself.

In [ ]:
os.environ["SELDON_SCHEDULE_HOST"] = f"0.0.0.0:{SCHED_LOCAL_PORT}"
os.environ["SELDON_INFER_HOST"]    = f"0.0.0.0:{INFER_LOCAL_PORT}"

print("SELDON_SCHEDULE_HOST =", os.environ["SELDON_SCHEDULE_HOST"])
print("SELDON_INFER_HOST    =", os.environ["SELDON_INFER_HOST"])
print()
print("To use the same values in an external shell, run:")
print(f'  export SELDON_SCHEDULE_HOST={os.environ["SELDON_SCHEDULE_HOST"]}')
print(f'  export SELDON_INFER_HOST={os.environ["SELDON_INFER_HOST"]}')

## 6. Verification

### 6a. Control plane (`seldon model list`)

Should return a table (possibly empty). Any RPC error means the scheduler port-forward isn't working.

In [ ]:
!seldon model list

### 6b. Data plane (raw HTTP through envoy)

Does not require any model to be loaded — we just want a meaningful HTTP response from envoy.

In [ ]:
rc, out, err = run(
    f"curl -sS -o /dev/null -w '%{{http_code}}' "
    f"http://0.0.0.0:{INFER_LOCAL_PORT}/v2/health/live"
)
print(f"GET /v2/health/live -> HTTP {out}")
if rc != 0:
    print("curl error:", err)

### 6c. End-to-end inference (optional)

If you already have an `iris` model loaded (e.g. from another sample notebook), this should print `Success: map[:iris_1::5]` or similar. Otherwise it will return an error — that's expected.

In [ ]:
!seldon model infer iris -i 5 \
  '{"inputs": [{"name": "predict", "shape": [1, 4], "datatype": "FP32", "data": [[1, 2, 3, 4]]}]}'

## Teardown (optional)

Run this cell to stop the port-forwards started by this notebook.

In [ ]:
for svc, lport in [(SCHEDULER_SVC, SCHED_LOCAL_PORT), (MESH_SVC, INFER_LOCAL_PORT)]:
    killed = kill_port_forward(svc, lport)
    print(f"stopped {killed} port-forward(s) for svc/{svc} (local :{lport})")